In [1]:
import torch 
import numpy as np 
import os, sys
from pathlib import Path
import importlib
import IPython.display as ipd

import src.saddler_w_gains_lightning as lm 
import yaml
from pytorch_lightning import Trainer, seed_everything
sys.path.append(str(Path.cwd().parent))
sys.path.append('phaselocknet_torch')
from phaselocknet_torch import phaselocknet_model
from phaselocknet_torch import util
import json 
import src.audio_transforms as at
import src.saddler_w_gains_lightning as  lm
import src.spatial_attn_architecture as arch


importlib.reload(phaselocknet_model)
importlib.reload(util)

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline
import matplotlib.pyplot as plt


In [2]:
### Dev with torch module 

importlib.reload(lm)
importlib.reload(arch)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

config_path = "config/binaural_attn/word_task_v10_saddler_backbone_learned_gains.yaml"
config = yaml.load(open(config_path, "r"), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 12


model = lm.BackBoneModule(config)

trainer.fit(model)

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /om2/user/imgriff/conda_envs/pytorch_2/lib/python3.1 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type  

[load_model_checkpoint] /om2/user/msaddler/auditoryutil/models/singletask/batchnorm_arch0_0000_taskW/ckpt_BEST.pt
Using dataset BinauralAttentionDataset
Sanity Checking: |          | 0/? [00:00<?, ?it/s]Using v06 dataset
/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v10
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
49 files in val concat dataset


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=1` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  1.04it/s]

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:77: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 12. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Using v06 dataset                                                          
/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v10
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
472 files in train concat dataset


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=1` in the `DataLoader` to improve performance.


len training set = 196352
Epoch 0:   0%|          | 29/196352 [00:16<31:54:39,  1.71it/s, train_loss_step=9.320, grad_norm=0.265]

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
model.model.gains.parameters()

<generator object Module.parameters at 0x153ee1298120>

In [ ]:
cue, _, scene, labels = next(iter(model.train_dataloader()))



Using v06 dataset
/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v10
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
472 files in train concat dataset
len training set = 196352


In [ ]:
cue.permute(0, 2, 1).shape, scene.permute(0, 2, 1).shape, labels.shape

(torch.Size([12, 125000, 2]), torch.Size([12, 125000, 2]), torch.Size([12]))